In [3]:
"""
CASA0006 - Data Preprocessing Validation Tests
"""

import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

# Load processed files
print("Loading processed files...")
collisions = pd.read_csv(PROCESSED_DIR / "london_collisions_2022_2023.csv")
casualties = pd.read_csv(PROCESSED_DIR / "london_casualties_2022_2023.csv")
vehicles = pd.read_csv(PROCESSED_DIR / "london_vehicles_2022_2023.csv")

print(f"\n{'='*60}")
print("VALIDATION TESTS")
print(f"{'='*60}")

# ========== TEST 1: Row count consistency ==========
print("\n[TEST 1] Row count consistency check")
expected = {
    'collisions': 46416,
    'casualties': 53469,
    'vehicles': 84319  # 42,783 + 41,536
}
actual = {
    'collisions': len(collisions),
    'casualties': len(casualties),
    'vehicles': len(vehicles)
}

for table, exp in expected.items():
    act = actual[table]
    status = "✓ PASS" if act == exp else "✗ FAIL"
    print(f"  {status} | {table}: expected {exp:,}, got {act:,}")

# ========== TEST 2: Geographic filter integrity ==========
print("\n[TEST 2] Geographic filter integrity (London only)")
unique_forces = collisions['police_force'].unique()
expected_forces = {1, 48}
actual_forces = set(unique_forces)

if actual_forces.issubset(expected_forces):
    print(f"  ✓ PASS | Only London police forces present: {sorted(actual_forces)}")
else:
    extra = actual_forces - expected_forces
    print(f"  ✗ FAIL | Non-London forces found: {extra}")

# Year coverage
unique_years = sorted(collisions['collision_year'].unique())
print(f"  Years in data: {unique_years}")
if set(unique_years) == {2022, 2023}:
    print(f"  ✓ PASS | Both years present, no leakage")
else:
    print(f"  ✗ FAIL | Unexpected years")

# ========== TEST 3: Referential integrity ==========
print("\n[TEST 3] Referential integrity (no orphan records)")

# Every casualty's collision_index should exist in collisions
collision_indices = set(collisions['collision_index'])
casualty_orphans = ~casualties['collision_index'].isin(collision_indices)
vehicle_orphans = ~vehicles['collision_index'].isin(collision_indices)

print(f"  Casualties: {casualty_orphans.sum()} orphan records (should be 0)")
print(f"  Vehicles:   {vehicle_orphans.sum()} orphan records (should be 0)")

if casualty_orphans.sum() == 0 and vehicle_orphans.sum() == 0:
    print(f"  ✓ PASS | All child records linked to a parent collision")
else:
    print(f"  ✗ FAIL | Orphan records detected")

# ========== TEST 4: Binary target correctness ==========
print("\n[TEST 4] Binary target variable correctness")

# severity_binary should be 1 if collision_severity ∈ {1,2}, else 0
expected_binary = collisions['collision_severity'].isin([1, 2]).astype(int)
mismatches = (collisions['severity_binary'] != expected_binary).sum()

if mismatches == 0:
    print(f"  ✓ PASS | severity_binary correctly derived from collision_severity")
else:
    print(f"  ✗ FAIL | {mismatches} rows have incorrect binary encoding")

# Verify class proportions
slight_pct = 100 * (collisions['severity_binary'] == 0).sum() / len(collisions)
serious_pct = 100 * (collisions['severity_binary'] == 1).sum() / len(collisions)
print(f"  Class distribution: Slight={slight_pct:.1f}%, Serious/Fatal={serious_pct:.1f}%")

# ========== TEST 5: No data leakage ==========
print("\n[TEST 5] No duplicate collision indices")

dup_count = collisions['collision_index'].duplicated().sum()
if dup_count == 0:
    print(f"  ✓ PASS | All collision indices are unique")
else:
    print(f"  ✗ FAIL | {dup_count} duplicate collision indices found")

# ========== TEST 6: Spot check (manual verification) ==========
print("\n[TEST 6] Spot check - random collision details")
sample = collisions.sample(n=3, random_state=42)
for idx, row in sample.iterrows():
    col_idx = row['collision_index']
    n_casualties_in_table = (casualties['collision_index'] == col_idx).sum()
    n_vehicles_in_table = (vehicles['collision_index'] == col_idx).sum()
    
    print(f"\n  Collision {col_idx}:")
    print(f"    Recorded n_casualties: {row['number_of_casualties']}")
    print(f"    Actual rows in casualties table: {n_casualties_in_table}")
    print(f"    Recorded n_vehicles: {row['number_of_vehicles']}")
    print(f"    Actual rows in vehicles table: {n_vehicles_in_table}")
    
    if row['number_of_casualties'] == n_casualties_in_table:
        print(f"    ✓ Casualty count matches")
    else:
        print(f"    ⚠️  Casualty count mismatch")
    
    if row['number_of_vehicles'] == n_vehicles_in_table:
        print(f"    ✓ Vehicle count matches")
    else:
        print(f"    ⚠️  Vehicle count mismatch")

print(f"\n{'='*60}")
print("Validation complete")
print(f"{'='*60}")

Loading processed files...

VALIDATION TESTS

[TEST 1] Row count consistency check
  ✓ PASS | collisions: expected 46,416, got 46,416
  ✓ PASS | casualties: expected 53,469, got 53,469
  ✓ PASS | vehicles: expected 84,319, got 84,319

[TEST 2] Geographic filter integrity (London only)
  ✓ PASS | Only London police forces present: [np.int64(1), np.int64(48)]
  Years in data: [np.int64(2022), np.int64(2023)]
  ✓ PASS | Both years present, no leakage

[TEST 3] Referential integrity (no orphan records)
  Casualties: 0 orphan records (should be 0)
  Vehicles:   0 orphan records (should be 0)
  ✓ PASS | All child records linked to a parent collision

[TEST 4] Binary target variable correctness
  ✓ PASS | severity_binary correctly derived from collision_severity
  Class distribution: Slight=84.0%, Serious/Fatal=16.0%

[TEST 5] No duplicate collision indices
  ✓ PASS | All collision indices are unique

[TEST 6] Spot check - random collision details

  Collision 2022010410780:
    Recorded n_ca

In [ ]:
import pandas as pd
from pathlib import Path

print("=" * 70)
print("POST-PROCESSING VALIDATION")
print("=" * 70)

processed_dir = Path(r"D:\0.CASA_USS\CASA0006-London-Road-Safety\data\processed")

collisions = pd.read_csv(processed_dir / "london_collisions_2022_2023.csv")
casualties = pd.read_csv(processed_dir / "london_casualties_2022_2023.csv")
vehicles = pd.read_csv(processed_dir / "london_vehicles_2022_2023.csv")

print(f"\n[FILE SIZES]")
for file in processed_dir.glob("*.csv"):
    size_mb = file.stat().st_size / (1024 ** 2)
    status = "✓" if size_mb < 100 else "✗ TOO LARGE"
    print(f"  {file.name:40s} {size_mb:6.1f} MB  {status}")

print(f"\n[ROW COUNTS]")
print(f"  Collisions:  {len(collisions):,}")
print(f"  Casualties:  {len(casualties):,}")
print(f"  Vehicles:    {len(vehicles):,}")

print(f"\n[BOROUGH DATA QUALITY]")
print(f"  Column 'borough_name' exists: {'✓' if 'borough_name' in collisions.columns else '✗ MISSING'}")

if 'borough_name' in collisions.columns:
    borough_dist = collisions['borough_name'].value_counts()
    
    print(f"\n  Borough distribution:")
    print(f"    Total unique boroughs: {len(borough_dist)}")
    print(f"    Unknown: {borough_dist.get('Unknown', 0):,} ({100*borough_dist.get('Unknown',0)/len(collisions):.2f}%)")
    
    print(f"\n  Top 10 boroughs by collision count:")
    for i, (borough, count) in enumerate(borough_dist.head(10).items(), 1):
        pct = 100 * count / len(collisions)
        print(f"    {i:2d}. {borough:30s} {count:5,} ({pct:5.2f}%)")

print(f"\n[SEVERITY BINARY DISTRIBUTION]")
if 'severity_binary' in collisions.columns:
    severity_dist = collisions['severity_binary'].value_counts()
    print(f"  Slight (0):        {severity_dist.get(0, 0):,} ({100*severity_dist.get(0,0)/len(collisions):.1f}%)")
    print(f"  Serious/Fatal (1): {severity_dist.get(1, 0):,} ({100*severity_dist.get(1,0)/len(collisions):.1f}%)")
    print(f"  Imbalance ratio:   {severity_dist.get(0,0)/severity_dist.get(1,0):.2f}:1")
else:
    print("  ✗ severity_binary column MISSING")

print(f"\n[REFERENTIAL INTEGRITY]")
collision_ids = set(collisions['collision_index'])
casualty_ids = set(casualties['collision_index'])
vehicle_ids = set(vehicles['collision_index'])

orphan_casualties = len(casualty_ids - collision_ids)
orphan_vehicles = len(vehicle_ids - collision_ids)

print(f"  Orphan casualties (not in collisions): {orphan_casualties:,} {'✓' if orphan_casualties == 0 else '✗'}")
print(f"  Orphan vehicles (not in collisions):   {orphan_vehicles:,} {'✓' if orphan_vehicles == 0 else '✗'}")

print(f"\n[YEARS COVERAGE]")
print(f"  Years in collision data: {sorted(collisions['collision_year'].unique())}")
print(f"  Date range: {collisions['date'].min()} to {collisions['date'].max()}")

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

# Generate summary for README
print("\n### Summary for GitHub README:")
print(f"""
## Processed Data Summary

**Dataset coverage**: London (Metropolitan Police + City of London), 2022-2023

| File | Rows | Size | Description |
|------|------|------|-------------|
| london_collisions_2022_2023.csv | {len(collisions):,} | {processed_dir.joinpath('london_collisions_2022_2023.csv').stat().st_size / (1024**2):.1f} MB | Primary collision records with environmental features |
| london_casualties_2022_2023.csv | {len(casualties):,} | {processed_dir.joinpath('london_casualties_2022_2023.csv').stat().st_size / (1024**2):.1f} MB | Casualty-level data (age, severity, type) |
| london_vehicles_2022_2023.csv | {len(vehicles):,} | {processed_dir.joinpath('london_vehicles_2022_2023.csv').stat().st_size / (1024**2):.1f} MB | Vehicle-level data (type, manoeuvre) |

**Borough coverage**: {len(borough_dist) - 1} London boroughs ({100 - 100*borough_dist.get('Unknown',0)/len(collisions):.1f}% of collisions georeferenced)

**Severity distribution**:
- Slight: {100*severity_dist.get(0,0)/len(collisions):.1f}%
- Serious/Fatal: {100*severity_dist.get(1,0)/len(collisions):.1f}%
""")

POST-PROCESSING VALIDATION

[FILE SIZES]
  london_casualties_2022_2023.csv             4.3 MB  ✓
  london_collisions_2022_2023.csv             9.3 MB  ✓
  london_vehicles_2022_2023.csv               9.0 MB  ✓

[ROW COUNTS]
  Collisions:  46,416
  Casualties:  53,469
  Vehicles:    84,319

[BOROUGH DATA QUALITY]
  Column 'borough_name' exists: ✓

  Borough distribution:
    Total unique boroughs: 34
    Unknown: 44 (0.09%)

  Top 10 boroughs by collision count:
     1. Westminster                    2,516 ( 5.42%)
     2. Southwark                      2,211 ( 4.76%)
     3. Tower Hamlets                  2,052 ( 4.42%)
     4. Wandsworth                     2,038 ( 4.39%)
     5. Croydon                        2,025 ( 4.36%)
     6. Waltham Forest                 1,993 ( 4.29%)
     7. Greenwich                      1,941 ( 4.18%)
     8. Brent                          1,866 ( 4.02%)
     9. Kensington and Chelsea         1,861 ( 4.01%)
    10. Ealing                         1,760 ( 3.